# V9.2a: ConcatScan From Scratch

**Goal:** Verify that text guidance is useful when backbone is NOT pretrained.

Train ConcatScan (Mamba-native fusion) from scratch on BraTS2020 (295 cases).
No BraTS2021 pretraining. If text delta > 0, proves the problem is
same-domain pretraining redundancy, not the fusion method.

| Version | Fusion | Training | Mean Dice | Text Delta |
|---------|--------|----------|-----------|------------|
| V5.0 | SeqCA | from scratch | 0.8479 | +0.55% |
| V8.0 | SeqCA | pretrained | 0.8753 | 0.00% |
| V9.1 | ConcatScan | pretrained | 0.8719 | -0.01% |
| **V9.2a** | **ConcatScan** | **from scratch** | **?** | **?** |

Key changes vs V9.1:
- No `--resume` (from scratch)
- Higher LR (1e-4 vs 5e-5)
- More epochs (300 vs 100)
- More patience (50 vs 30)
- alignment_weight=0.0 (clean experiment, one variable)


In [1]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo 'No GPU'

!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob, threading
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0:
            break
        print(f'Clone attempt {attempt} failed')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else:
        raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

# BraTS2020 + TextBraTS
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
print(f'BraTS2020 data: {len([d for d in os.listdir(DATA_DIR) if d.startswith("BraTS")])} cases')

def sync_and_tag(tag):
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    best = os.path.join(local_ckpt, 'best.pth')
    if os.path.exists(best):
        shutil.copy2(best, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
        print(f'Tagged: best_{tag}.pth')
    last = os.path.join(local_ckpt, 'last.pth')
    if os.path.exists(last):
        shutil.copy2(last, os.path.join(DRIVE_CKPT, f'last_{tag}.pth'))
    print(f'Synced to {DRIVE_CKPT}')

# Background auto-sync
_sync_stop = threading.Event()
_sync_track = {}

def _file_is_stable(path, wait=3):
    try:
        s1 = os.path.getsize(path)
        time.sleep(wait)
        s2 = os.path.getsize(path)
        return s1 == s2 and s1 > 0
    except OSError:
        return False

def _bg_sync_loop():
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    while not _sync_stop.is_set():
        _sync_stop.wait(120)
        if _sync_stop.is_set():
            break
        try:
            files = glob.glob(os.path.join(local_ckpt, '*.pth'))
            synced = 0
            for f in files:
                mt = os.path.getmtime(f)
                name = os.path.basename(f)
                if name not in _sync_track or _sync_track[name] < mt:
                    if not _file_is_stable(f):
                        continue
                    shutil.copy2(f, os.path.join(DRIVE_CKPT, name))
                    _sync_track[name] = mt
                    synced += 1
            if synced > 0:
                print(f'[AutoSync] {synced} checkpoint(s) synced to Drive')
        except Exception as e:
            print(f'[AutoSync] Warning: {e}')

_sync_thread = threading.Thread(target=_bg_sync_loop, daemon=True)
_sync_thread.start()
print('Background auto-sync started (every 2 min)')
print('Setup complete')


Mounted at /content/drive
Sun Apr  5 05:08:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   44C    P0             71W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

## Training: ConcatScan From Scratch

300 epochs, no pretrained checkpoint, LR=1e-4


In [2]:
import os, glob, shutil
os.chdir(REPO_DIR)

os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

# Use version-specific checkpoint name, not shared last.pth
V92A_LAST = 'last_V9.2a.pth'
V92A_BEST = 'best_V9.2a.pth'

resume_args = ''
ckpt_dir = os.path.join(REPO_DIR, 'checkpoints')
os.makedirs(ckpt_dir, exist_ok=True)

drive_last = os.path.join(DRIVE_CKPT, V92A_LAST)
drive_best = os.path.join(DRIVE_CKPT, V92A_BEST)

# Check if already complete
if os.path.exists(drive_best):
    import torch
    try:
        ckpt = torch.load(drive_best, map_location='cpu', weights_only=False)
        ep = ckpt.get('epoch', -1)
        del ckpt
        if ep >= 200:
            print(f'V9.2a already complete (epoch {ep}). Skip to evaluation.')
            raise SystemExit(0)
    except SystemExit:
        raise
    except Exception:
        pass

# Resume from version-specific checkpoint
if os.path.exists(drive_last):
    import torch
    try:
        ckpt = torch.load(drive_last, map_location='cpu', weights_only=False)
        ep = ckpt.get('epoch', -1)
        bd = ckpt.get('best_dice', 0)
        del ckpt
        shutil.copy2(drive_last, os.path.join(ckpt_dir, 'last.pth'))
        for name in ['best.pth', 'best_no_text.pth']:
            src = os.path.join(DRIVE_CKPT, name)
            if os.path.exists(src):
                shutil.copy2(src, os.path.join(ckpt_dir, name))
        resume_args = '--resume checkpoints/last.pth'
        print(f'Resuming V9.2a from epoch {ep}, best_dice={bd:.4f}')
    except Exception as e:
        print(f'Could not load {V92A_LAST} ({e}), starting fresh')
else:
    for f in glob.glob(os.path.join(ckpt_dir, '*.pth')):
        os.remove(f)
    print('Starting V9.2a from scratch')

print('V9.2a: ConcatScan from scratch (no pretrain)')
!python -u train.py \
    --config configs/autoresearch/V9.2a_concat_scratch.yaml \
    --no-text-ratio 0.15 \
    --grad-accum 2 \
    {resume_args}

# Save version-specific checkpoints
local_last = os.path.join(ckpt_dir, 'last.pth')
if os.path.exists(local_last):
    shutil.copy2(local_last, os.path.join(DRIVE_CKPT, V92A_LAST))
sync_and_tag('V9.2a')
print('V9.2a complete!')


In [2]:
# === Emergency Sync ===
import shutil, glob, os, subprocess
local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
files = glob.glob(os.path.join(local_ckpt, '*.pth'))
if not files:
    print('No local checkpoints')
else:
    for f in sorted(files):
        name = os.path.basename(f)
        shutil.copy2(f, os.path.join(DRIVE_CKPT, name))
    subprocess.run(['sync'], check=True)
    print(f'{len(files)} files synced')


No local checkpoints


## Evaluation


In [3]:
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_V9.2a.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Evaluating: {ckpt}')

CONFIG = 'configs/autoresearch/V9.2a_concat_scratch.yaml'

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    print()
    print('=' * 60)
    print(name)
    print('=' * 60)
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', CONFIG,
           '--checkpoint', ckpt,
           '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(ret.stdout)
    if ret.returncode != 0:
        print(f'ERROR: {ret.stderr[-500:]}')

print()
print('Key comparison:')
print('  V5.0 (SeqCA, scratch):     Mean=0.8479, delta=+0.55%')
print('  V8.0 (SeqCA, pretrained):  Mean=0.8753, delta=0.00%')
print('  V9.1 (ConcatScan, pretrained): Mean=0.8719, delta=-0.01%')
print('  V9.2a (ConcatScan, scratch):   Mean=?, delta=?')


Evaluating: /content/drive/MyDrive/TextMamba3D/checkpoints/best_V9.2a.pth

text+TTA
Loaded checkpoint: epoch=194, best_dice=0.8788700766033597
TextBraTS test: 95 samples

Evaluating 95 cases (test split)
Sliding window: patch=(128, 128, 128), overlap=0.5, text=True
TTA: 8-fold flip ensemble ENABLED

  BraTS20_Training_328: Dice=0.4464 (ET=0.2647, TC=0.1905, WT=0.8838) HD95_ET=51.38
  BraTS20_Training_028: Dice=0.7598 (ET=0.5959, TC=0.8455, WT=0.8379) HD95_ET=2.24
  BraTS20_Training_289: Dice=0.5492 (ET=0.0000, TC=0.7064, WT=0.9413) HD95_ET=nan
  BraTS20_Training_231: Dice=0.9338 (ET=0.9129, TC=0.9325, WT=0.9561) HD95_ET=1.00
  BraTS20_Training_261: Dice=0.6819 (ET=0.3016, TC=0.7928, WT=0.9513) HD95_ET=29.14
  BraTS20_Training_163: Dice=0.9212 (ET=0.8519, TC=0.9392, WT=0.9725) HD95_ET=1.00
  BraTS20_Training_345: Dice=0.6161 (ET=0.8494, TC=0.2547, WT=0.7443) HD95_ET=6.00
  BraTS20_Training_139: Dice=0.9021 (ET=0.8942, TC=0.9282, WT=0.8839) HD95_ET=1.00
  BraTS20_Training_063: Dice=0.665